# Step 3b: Group 10's NRMS-inspired model, corrected reimplementation

This notebook rebuilds the strongest model from the assigned past project ([bt422210/BT4222-Group-10](https://github.com/bt422210/BT4222-Group-10), commit `3007ee3`) and runs it on our split, so it can be compared with our models on identical candidates and metrics.

It is a **corrected reimplementation**, not an exact reproduction: their pipeline stores intermediate files on their own Google Drive, so we rebuild every input from the raw MIND files with their recipes, and we fix the defects listed below.

### What their model is

Despite the name, this is not standard NRMS (which applies self-attention over title words). It is a hybrid ranker:

```
news article ─┬─ text tokens:   [MiniLM title ⊕ abstract (768-d)] + [category-cluster embedding]  → 2-token self-attention → 128-d
              └─ entity tokens: [KG title, KG abstract (100-d)] + [NER title, NER abstract (384-d)] → masked 4-token self-attention → 128-d
              → fusion layer → news vector (128-d)

user          ─ last 10 clicked articles → news encoder → self-attention block → additive attention → user vector (128-d)
                (learnable default vector when the history is empty)

scorer (MLP)  ─ [user vector, news vector, user numeric features, 6 user categorical features,
                 50 TF-IDF topics of the candidate, hour sin/cos, habit score, article recency] → 128 → 64 → 32 → 1
```

Training: one click plus four non-clicked candidates sampled from the same impression, softmax cross-entropy over the five, Adam (lr 5×10⁻⁴), batch 32, 10 epochs. We use their **Test_2 configuration (2 heads, dropout 0.1)**, the one with the best validation score among their four runs.

### Kept exactly as Group 10 built it

| Component | Recipe |
|---|---|
| Split | Same as theirs and ours: train Nov 9–12, validation Nov 13, test Nov 14 |
| Text vectors | `all-MiniLM-L6-v2` on title and on abstract (title when missing), each normalised, concatenated, renormalised |
| Entity vectors | Mean KG embedding of title / abstract WikidataIds; spaCy `en_core_web_sm` entity phrases encoded with MiniLM; four presence flags |
| Categories | Subcategories with ≥ 75th-percentile size keep their label, else `<category>_general`; labels under 100 articles → `miscellaneous`; 8 largest split into 3 KMeans clusters |
| Topics | TF-IDF (1–2-grams, 1,000 terms) on titles → 50 SVD components |
| User features | History length (log, scaled), category diversity and concentration, top-3 and recent favourite category, diversity cluster (KMeans 3), interest cluster (MiniBatchKMeans 4), all fitted on training rows |
| Negatives | 4 per click, from the same impression, with replacement when fewer than 4 exist |
| Architecture and hyperparameters | Layer for layer, see `shared/nrms_g10.py` |

Category merging, clustering and topics are fitted on the whole article catalogue, as Group 10 did. They use article text only, never labels or clicks.

### Corrected

| # | Issue in Group 10's code | Evidence | Correction |
|---|---|---|---|
| 1 | Training recency scores do not line up with the candidates | `HybridNewsDataset.__getitem__` re-shuffles candidates with a new permutation but takes `recency_scores[idx]` in the context table's own order | Recency is computed per candidate row and travels with it |
| 2 | Context features are fetched by row position from tables that were re-sorted | `3_feature_engineering` sorts the context tables by user and time before saving; user features keep the original order; the model indexes both by position. Their parquet conversion is not in the repo, so this is *probable*, not confirmed | Every input is joined by `impression_id` or news id |
| 3 | Recency ignores exposures outside the sampled training rows | First-seen time comes from the 1+4 sampled training candidates only; any article first shown on Nov 13 or 14 keeps recency 1.0 however old it is | First exposure anywhere in the log strictly before the impression |
| 4 | Habit score resets at each split boundary | Previous visit is looked up within the same split | Previous impression anywhere in the log strictly before t |
| 5 | Test uses the last epoch | A best epoch is recorded but its weights are not restored | Best validation epoch is restored before scoring test |
| 6 | No random seeds | None set anywhere | Seeds fixed |
| 7 | Evaluation not comparable | Pooled AUC, MRR from the first click only, and stable sorting that favours clicks on ties | MIND-standard per-impression metrics with a seeded tie-break; their legacy metrics are also shown for comparison |
| 8 | Headline numbers were training scores on 1+4 sampled slates | Their summary table | Train, validation and test are all scored on full slates |

> Metrics here come from `shared/eval_interim.py`, a stand-in for tuning until the team evaluator exists. The numbers in the final report come from `step6_evaluation_diagnostics.ipynb`.

## 1. Setup

In [ ]:
import os, sys, subprocess, time

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Bryng23/BT4222-Group-11.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/BT4222-Group-11"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    DATA_ROOT = "/content/drive/MyDrive/BT4222_G11"
else:
    REPO_DIR = os.getcwd()
    DATA_ROOT = os.path.join(REPO_DIR, "data")
sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from shared import schema as S
from shared.predictions import write_predictions
from shared.nrms_inputs import build_inputs
from shared import nrms_g10 as N
from shared import eval_interim as E      # swap for the team evaluator once it exists

DATA_KIND = os.environ.get("G11_DATA_KIND", "REAL")   # SYNTHETIC only for local development
if DATA_KIND == "REAL":
    import kagglehub
    RAW_DIR = os.path.join(kagglehub.dataset_download("arashnic/mind-news-dataset"), "MINDsmall_train")
    FEATURE_ROOT = os.path.join(DATA_ROOT, "features", f"v{S.SCHEMA_VERSION}")
    TEXT_METHOD, NER_METHOD, CAT_KWARGS = "minilm", "spacy", {}
    try:
        import spacy; spacy.load("en_core_web_sm")
    except OSError:
        subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
else:
    FEATURE_ROOT = os.path.join(DATA_ROOT, "features", f"synthetic_v{S.SCHEMA_VERSION}", "learnable")
    RAW_DIR = os.path.join(FEATURE_ROOT, "raw")
    # offline stand-ins: no model download, and category thresholds scaled to a 1,200-article catalogue
    TEXT_METHOD, NER_METHOD, CAT_KWARGS = "tfidf_svd", "none", {"merge_threshold": 20}
PRED_ROOT = os.path.join(DATA_ROOT, "predictions")
CACHE_DIR = os.path.join(DATA_ROOT, "nrms_cache", DATA_KIND.lower())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 4222

CONFIG = dict(num_heads=2, dropout=0.1, user_heads=2, user_dropout=0.1)   # Group 10 Test_2
TRAIN = dict(epochs=10, lr=5e-4, batch_size=32)
SELECT = "ndcg@5"   # Group 10's own checkpoint criterion
MODEL_ID = "nrms_g10"
print(f"DATA_KIND={DATA_KIND} | device={DEVICE} | raw={RAW_DIR}")

## 2. Candidate rows and labels from the shared metadata tables

Every prediction must carry the shared `row_id`, so the candidate lists come from `meta_{split}.parquet` rather than from re-parsing `behaviors.tsv`.

In [ ]:
META_COLS = ["row_id", "impression_id", "news_id", "time", "label", "is_cold_10"]
meta = {s: pd.read_parquet(S.meta_path(FEATURE_ROOT, s), columns=META_COLS) for s in S.SPLITS}
GROUP10_IMPRESSIONS = {"train": 95071, "val": 31624, "test": 30270}
pd.DataFrame({s: {"impressions": m["impression_id"].nunique(), "Group 10 impressions": GROUP10_IMPRESSIONS[s],
                  "candidate rows": len(m), "CTR": round(m["label"].mean(), 4)} for s, m in meta.items()}).T

## 3. Build Group 10's inputs

The MiniLM and spaCy steps are the slow part (roughly 10–30 minutes on free Colab) and are cached on Drive, so later runs skip them. The printed counts can be checked against Group 10's notebooks: 79 merged categories, 95 sub-clustered labels, and 141,558 training samples.

In [ ]:
t0 = time.time()
inputs = build_inputs(RAW_DIR, meta, text_method=TEXT_METHOD, ner_method=NER_METHOD, device=DEVICE,
                      cache_dir=CACHE_DIR, cat_kwargs=CAT_KWARGS, seed=SEED)
print(f"built in {time.time() - t0:.0f}s")
for split in S.SPLITS:   # every candidate row is attached to its own impression
    assert (inputs["impressions"]["impression_id"][inputs["splits"][split]["imp"]] == meta[split]["impression_id"].to_numpy()).all()
flags = inputs["news"]["flags"].mean(0)
pd.Series({**inputs["info"], "user vocab sizes": inputs["user_vocab_sizes"],
           "share with KG title / KG abstract / NER title / NER abstract": np.round(flags, 3).tolist()})

## 4. Train with Group 10's loop, selecting the epoch on validation

In [ ]:
N.set_seed(SEED)
model = N.NRMSG10(inputs, **CONFIG).to(DEVICE)
print(f"trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

SHOW = ["auc", "mrr", "ndcg@5", "ndcg@10", "cold_recall@10"]
val_meta, val_split = meta["val"], inputs["splits"]["val"]
def val_fn(m):
    res = E.evaluate(val_meta, N.score_split(m, val_split))
    return {k: res[k] for k in SHOW}

history, best_epoch = N.train(model, inputs["train_samples"], seed=SEED, val_fn=val_fn, select_metric=SELECT, **TRAIN)
print(f"best validation {SELECT} at epoch {best_epoch}; those weights are now loaded")
history.round(4)

## 5. Score every split with the selected epoch

Train, validation and test are scored on their full slates and written as prediction files. Group 10's own legacy metrics (pooled AUC, first-click MRR) are computed alongside, so this run can be lined up against their saved test results for the same configuration.

In [ ]:
def legacy_metrics(m, score):
    """Group 10's definitions: AUC pooled over all rows; MRR of the first click per impression."""
    d = m[["impression_id", "label", "row_id"]].assign(s=score, k=E.tie_key(m["row_id"].to_numpy()))
    d = d.sort_values(["impression_id", "s", "k"], ascending=[True, False, True])
    d["rank"] = d.groupby("impression_id").cumcount() + 1
    first = d[d["label"] == 1].groupby("impression_id")["rank"].min()
    return {"pooled_auc": roc_auc_score(m["label"], score), "first_click_mrr": float((1 / first).mean())}

config_id = f"heads={CONFIG['num_heads']},dropout={CONFIG['dropout']},epoch={best_epoch}"
rows = []
for split in S.SPLITS:
    sc = N.score_split(model, inputs["splits"][split])
    write_predictions(meta[split]["row_id"], sc, root=PRED_ROOT, data_kind=DATA_KIND, model_id=MODEL_ID,
                      config_id=config_id, split=split, schema_version=S.SCHEMA_VERSION,
                      extra={"selected_on": "val", "selection_metric": SELECT, "group10_config": "Test_2"})
    res = E.evaluate(meta[split], sc)
    rows.append({"split": split, **{k: res[k] for k in SHOW + ["n_cold_click_impressions"]}, **legacy_metrics(meta[split], sc)})
results = pd.DataFrame(rows).set_index("split")
results.round(4)

In [ ]:
group10_test = pd.Series({"pooled_auc": 0.7046, "first_click_mrr": 0.2951, "ndcg@5": 0.2699, "ndcg@10": 0.3319},
                         name="Group 10 saved test (Test_2, last epoch)")
ours = results.loc["test", group10_test.index].rename("this reimplementation, test")
pd.concat([group10_test, ours], axis=1).round(4)

## 6. Reading the results

* The last table is the fidelity check. Close numbers mean the rebuilt model behaves like theirs. A gap can come from the corrections above (especially recency alignment and checkpoint selection), and should be described rather than tuned away.
* All further comparisons with our models use the MIND-standard columns, not the legacy ones.
* Their model has no article-level behavioural inputs (click counts or CTR); its only time signal is recency. Its Cold Recall@10 is therefore the natural point of comparison for our content-transfer features.